# 00 · Ambiente, dati e protocollo

**Tesi:** *Valutazione di una strategia di market making mediante dati LOB L2, L3 e L4.*

**Obiettivo.** Una strategia di market making passivo viene valutata con lo stesso codice ma con modelli di esecuzione diversi: tre regole che deducono le esecuzioni dai soli prezzi aggregati (dati L2) e un replay che ricostruisce la coda reale degli ordini (dati L3 e L4).

**Domanda di ricerca.** Quanto cambia la valutazione di una strategia di market making passivo quando le esecuzioni stimate con regole L2 vengono confrontate con un replay order-level che considera la posizione nella coda?

**Domanda secondaria.** Il filtro di bassa volatilità riduce le perdite e il rischio di inventario rispetto alla strategia sempre attiva?

| Dataset | Informazione | Unità | Uso principale | Limite |
|---|---|---|---|---|
| FI-2010 | L2 normalizzato | titolo-giorno | baseline L2 | nessuna coda, P&L non monetario |
| Nasdaq ITCH | L3/MBO | titolo-giorno | risultato principale | ordini nascosti e market impact esclusi |
| Hyperliquid | L4 + trade | ora | robustezza | book diff senza timestamp |

I termini tecnici sono spiegati nel **glossario** in fondo al notebook 06.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/market_making_tesi").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from market_making_tesi import report as R
from market_making_tesi.report import it, pct
R.setup_plots()
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
np.random.seed(0)
import platform
from importlib.metadata import version
environment = {"python": platform.python_version(), "sistema": platform.platform(),
               **{n: version(n) for n in ["numpy", "pandas", "pyarrow", "matplotlib"]}}
(ROOT / "results" / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
display(pd.Series(environment, name="versione").to_frame())

,versione
python,3.14.6
sistema,Windows-11-10.0.26300-SP0
numpy,2.5.4
pandas,3.0.6
pyarrow,26.0.0
matplotlib,3.11.2


## 1. Da dove vengono i numeri di questi notebook

I dati grezzi (circa 33 GB) non sono nel repository. I risultati si ottengono in due passaggi:

1. **dai dati grezzi:** `scripts/download_data.py`, `mm-backtest ingest` e `mm-backtest pipeline` producono i file CSV in `results/`;
2. **nei notebook:** 02, 04, 05 e 06 leggono soltanto quei CSV; 01 e 03 leggono anche i dati intermedi in `data/interim/` per grafici e controlli, e se mancano mostrano i grafici salvati.

La cella seguente indica quali dati sono presenti in questa esecuzione e quando sono stati prodotti i risultati.

In [2]:
import datetime as dt
status = pd.DataFrame({
    "presente": {"data/raw (dati grezzi)": R.data_available(ROOT, "data", "raw"),
                 "data/interim (dati intermedi)": R.data_available(ROOT, "data", "interim"),
                 "results/test.csv (risultati della pipeline)": R.data_available(ROOT, "results", "test.csv")},
})
modified = (ROOT / "results" / "test.csv").stat().st_mtime
display(status)
display(Markdown(f"I risultati di test caricati dai notebook sono stati scritti dalla pipeline il "
                 f"{dt.datetime.fromtimestamp(modified):%d/%m/%Y alle %H:%M}. In questa esecuzione i notebook "
                 "**non rieseguono** la pipeline: caricano i CSV."))

,presente
data/raw (dati grezzi),True
data/interim (dati intermedi),True
results/test.csv (risultati della pipeline),True


I risultati di test caricati dai notebook sono stati scritti dalla pipeline il 16/10/2026 alle 13:31. In questa esecuzione i notebook **non rieseguono** la pipeline: caricano i CSV.

## 2. I dati grezzi sono quelli dichiarati?

Ogni file scaricato è registrato in `data/manifest.json` con fonte, data di accesso e impronta SHA-256: se cambiasse anche un solo byte, l'impronta non corrisponderebbe. Se i dati grezzi sono presenti, la cella ricalcola le impronte (qualche minuto).

In [3]:
import hashlib
manifest = json.loads((ROOT / "data" / "manifest.json").read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()

rows = []
for key, entry in sorted(manifest.items()):
    path = ROOT / key
    present = path.exists()
    ok = present and path.stat().st_size == entry["bytes"] and sha256(path) == entry["sha256"]
    rows.append({"dataset": key.split("/")[2], "GB": entry["bytes"] / 1e9, "presenti": present, "verificati": ok,
                 "fonte": entry["source"].split("/")[2], "accesso": entry["accessed"]})
files = pd.DataFrame(rows)
summary = files.groupby("dataset").agg(file=("GB", "size"), GB=("GB", "sum"), presenti=("presenti", "sum"),
                                       verificati=("verificati", "sum"), fonte=("fonte", "first"), accesso=("accesso", "max"))
display(summary.style.format({"GB": "{:.2f}"}))
if files.presenti.any():
    assert files[files.presenti].verificati.all(), "file alterati: rieseguire scripts/download_data.py"
else:
    display(Markdown("*Dati grezzi assenti: verifica degli hash non eseguita.*"))

,file,GB,presenti,verificati,fonte,accesso
dataset,,,,,,
fi2010,10,0.94,10,10,etsin.fairdata.fi,2026-10-14
hyperliquid,12,0.79,12,12,zenodo.org,2026-10-14
itch,7,31.01,7,7,emi.nasdaq.com,2026-10-14


## 3. La strategia in una pagina

La strategia è volutamente semplice e **identica in tutti gli esperimenti**; cambia solo il modo di decidere se i suoi ordini vengono eseguiti.

1. **Misura la volatilità** del prezzo medio (*midpoint*) su una finestra mobile.
2. **Opera solo se il mercato è calmo**: sopra una soglia cancella gli ordini, chiude l'inventario e aspetta.
3. **Quota una coppia**: un ordine di acquisto al miglior prezzo in acquisto (*bid*) e uno di vendita al miglior prezzo in vendita (*ask*). Se entrambi vengono eseguiti, guadagna lo *spread*.
4. **Chiude il rischio**: se dopo un certo tempo è stato eseguito un solo lato, cancella l'altro e chiude la posizione con un ordine a mercato.

| Componente | Riferimento | Cosa se ne riprende |
|---|---|---|
| Guadagno sullo spread, rischio di inventario | Ho e Stoll (1981); Avellaneda e Stoikov (2008) | Il market maker guadagna sulla differenza tra le due quote ma, detenendo inventario, è esposto ai movimenti del prezzo. |
| Filtro di volatilità | Avellaneda e Stoikov (2008) | Il rischio di inventario cresce con la varianza del prezzo. Qui, invece di allargare le quote come nel modello ottimo, si smette di quotare sopra una soglia. |
| Inventario limitato | Guéant, Lehalle e Fernandez-Tapia (2013) | Market making con un limite all'inventario; qui il limite è una sola quantità per lato. |
| Liquidazione a scadenza | Cartea, Jaimungal e Penalva (2015) | Le strategie con orizzonte finito chiudono l'inventario residuo pagando lo spread. |
| Perdita per selezione avversa | Glosten e Milgrom (1985) | Le esecuzioni arrivano spesso quando il prezzo sta per muoversi contro il market maker. |

La strategia **non implementa** il modello di Avellaneda e Stoikov: ne riprende solo l'idea che il rischio di inventario cresce con la volatilità.

In [4]:
from market_making_tesi.config import StrategyConfig
from market_making_tesi.pipeline import DATASETS, session_list
cfg = {n: StrategyConfig.from_json(ROOT / "config" / f"{n}.json") for n in ["fi2010", "itch", "hyperliquid"]}
display(pd.DataFrame({
    "quantità per lato": {n: f"{it(c.quote_size, 2 if c.quote_size < 1 else 0)} {u}" for (n, c), u in
                          zip(cfg.items(), ["unità", "azioni", "BTC"])},
    "inventario massimo": {n: it(c.max_abs_inventory, 2 if c.max_abs_inventory < 1 else 0) for n, c in cfg.items()},
    "volatilità misurata": {"fi2010": "per osservazione", "itch": "midpoint ogni secondo",
                            "hyperliquid": "per evento (dati senza orari)"},
}).rename(index=R.DATASET_LABELS))

,quantità per lato,inventario massimo,volatilità misurata
FI-2010,1 unità,1,per osservazione
Nasdaq ITCH,100 azioni,100,midpoint ogni secondo
Hyperliquid BTC,"0,01 BTC","0,01",per evento (dati senza orari)


## 4. Protocollo: calibrazione, validazione, test

Le sessioni sono divise **in ordine cronologico**, così i parametri non vengono scelti sugli stessi dati su cui si misura il risultato:

| Fase | A cosa serve |
|---|---|
| Calibrazione | stimare le soglie di volatilità |
| Validazione | scegliere finestra, soglia e durata degli ordini tra 12 combinazioni |
| Test | misurare il risultato, senza più modificare nulla |

Una *sessione* è una giornata (FI-2010, ITCH) o un'ora di negoziazione (Hyperliquid); una *titolo-sessione* è un titolo in una sessione.

In [5]:
rows = []
for name, spec in DATASETS.items():
    for session, symbol, _ in session_list(ROOT, spec):
        phase = "calibrazione" if session in spec.calibration else "validazione" if session in spec.validation else "test"
        rows.append({"dataset": R.DATASET_LABELS[name], "fase": phase, "sessione": session, "simbolo": symbol})
sessions = pd.DataFrame(rows)
order = {"calibrazione": 0, "validazione": 1, "test": 2}
table = (sessions.groupby(["dataset", "fase"]).agg(sessioni=("sessione", "nunique"), titolo_sessioni=("sessione", "size"),
                                                    dal=("sessione", "min"), al=("sessione", "max"))
         .reset_index().sort_values(["dataset", "fase"], key=lambda s: s.map(order) if s.name == "fase" else s))
display(table.set_index(["dataset", "fase"]))

sessioni  titolo_sessioni          dal  \
dataset         fase                                                   
FI-2010         calibrazione         1                5        day01   
                validazione          2               10        day02   
                test                 7               35        day04   
Hyperliquid BTC calibrazione         1                1  20251201T08   
                validazione          1                1  20251201T14   
                test                 4                4  20251202T08   
Nasdaq ITCH     calibrazione         1                3   2019-01-30   
                validazione          1                3   2019-03-27   
                test                 5               15   2019-07-30   

                                       al  
dataset         fase                       
FI-2010         calibrazione        day01  
                validazione         day03  
                test                day10  
Hyperliquid BTC calibrazione  20251201T08  
                validazione   20251201T14  
                test          20251203T14  
Nasdaq ITCH     calibrazione   2019-01-30  
                validazione    2019-03-27  
                test           2020-01-30

**Rischio di overfitting.** FI-2010 usa 1 giornata di calibrazione e 2 di validazione, ITCH 1 e 1, Hyperliquid 1 ora e 1 ora. Su un validation set così piccolo il confronto tra 12 combinazioni può premiare quella che funziona per caso (Bailey, Borwein, López de Prado e Zhu, 2017), soprattutto per ITCH e Hyperliquid. Per questo i notebook mostrano l'intera griglia di validazione e il test non viene mai usato per scegliere.

## 5. Il codice funziona?

I test automatici verificano ogni componente su casi costruiti a mano: lettura dei file, ricostruzione del book, regole di esecuzione, coda, latenza, contabilità, metriche e bootstrap per sessione. Poi il motore viene eseguito sui piccoli campioni di `data/samples`, che devono riprodurre valori noti.

In [6]:
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                       cwd=ROOT, capture_output=True, text=True, timeout=600)
log = check.stdout + check.stderr
(ROOT / "results" / "test_report.txt").write_text(log, encoding="utf-8")
assert check.returncode == 0, log
ran = next(line for line in log.splitlines() if line.startswith("Ran "))
from market_making_tesi.experiments import run_samples
samples, _ = run_samples(ROOT, ROOT / "results" / "samples")
assert samples.fill_maker.tolist() == [22, 5, 2, 2, 0]
R.show_box("Esito dei controlli", [
    f"{ran.split()[1]} test automatici superati.",
    "I cinque campioni di controllo riproducono i valori attesi.",
    "Dati grezzi verificati sul manifesto." if files.presenti.any() else "Dati grezzi non presenti: hash non verificati.",
])

> **Esito dei controlli**
>
> - 51 test automatici superati.
> - I cinque campioni di controllo riproducono i valori attesi.
> - Dati grezzi verificati sul manifesto.